
# ShrimpDiseaseImageBD YOLO Object Detection Benchmark — Under 12M Suite — RTX4090

Notebook này tải dataset Kaggle `nhanayai/shrimpdiseaseimagebd`, convert labels object detection sang canonical 2 lớp `BG` và `WSSV`, thêm Healthy ảnh negative/background, train **17 YOLO detection models dưới 12M parameters** bằng **native Ultralytics Python API**, resume/skip nếu bị lỗi giữa chừng, eval test split, tính thêm image-level diagnosis metrics và benchmark inference speed.

Models chính:
`yolov5nu.pt`, `yolov5su.pt`, `yolov5n6u.pt`, `yolov8n.pt`, `yolov8s.pt`, `yolov9t.pt`, `yolov9s.pt`, `yolov10n.pt`, `yolov10s.pt`, `yolo11n.pt`, `yolo11s.pt`, `yolo12n.pt`, `yolo12s.pt`, `yolov13n.pt`, `yolov13s.pt`, `yolo26n.pt`, `yolo26s.pt`.

Các model mới thêm so với bản 17-model:
- `yolov5n6u.pt`: under-12M và input-1280 family, đáng thử cho small objects/WSSV spots.
- `yolov8s.pt`: 11.2M, vừa dưới ngưỡng 12M.
- `yolov13n.pt`, `yolov13s.pt`: YOLOv13 N/S under-12M, Ultralytics-style repo/weights nếu môi trường hỗ trợ.

Không đưa YOLOv7-tiny/YOLOv6Lite vào suite chính của notebook này vì mục tiêu notebook là một pipeline Ultralytics-native `.pt`/supported model training. YOLOv7 không được Ultralytics hỗ trợ native train/inference trực tiếp; YOLOv6 trong Ultralytics chủ yếu là YAML training, không có COCO `.pt` pretrained tương đương trong cùng registry, nên không fair khi chỉ train 30 epochs.

Box-level classes:
```yaml
0: BG
1: WSSV
```

Healthy is not a detection class. Healthy images are negative samples with empty labels.


In [7]:

from pathlib import Path
import os, sys, json, time, shutil, random, subprocess, platform
from datetime import datetime

WORKDIR = Path("/home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2")
WORKDIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "nhanayai/shrimpdiseaseimagebd"
DATASET_OUT = WORKDIR / "datasets" / "shrimp_od_yolo_random_seed42"
RUNS_DIR = WORKDIR / "runs"
EXPORTS_DIR = WORKDIR / "exports"
REPORTS_DIR = WORKDIR / "reports"
for p in [RUNS_DIR, EXPORTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

EPOCHS = 30
IMG_SIZE = 1024
PATIENCE = 15
DEVICE = 0
WORKERS = 8
BATCH = -1
CACHE = "disk"       # change to False if disk space is limited
AMP = True
DETERMINISTIC = True
CLOSE_MOSAIC = 10

PRED_CONF = 0.25
PRED_IOU = 0.60
MAX_DET = 300

DO_EXPORT = False
EXPORT_FORMATS = ["onnx"]

MODEL_SPECS = [
    {"tag": "YOLOv5nu", "weights": "yolov5nu.pt"},
    {"tag": "YOLOv5su", "weights": "yolov5su.pt"},
    {"tag": "YOLOv5n6u", "weights": "yolov5n6u.pt"},
    {"tag": "YOLOv8n", "weights": "yolov8n.pt"},
    {"tag": "YOLOv8s", "weights": "yolov8s.pt"},
    {"tag": "YOLOv9-T", "weights": "yolov9t.pt"},
    {"tag": "YOLOv9-S", "weights": "yolov9s.pt"},
    {"tag": "YOLOv10n", "weights": "yolov10n.pt"},
    {"tag": "YOLOv10s", "weights": "yolov10s.pt"},
    {"tag": "YOLO11n", "weights": "yolo11n.pt"},
    {"tag": "YOLO11s", "weights": "yolo11s.pt"},
    {"tag": "YOLO12n", "weights": "yolo12n.pt"},
    {"tag": "YOLO12s", "weights": "yolo12s.pt"},
    {"tag": "YOLOv13-N", "weights": "yolov13n.pt"},
    {"tag": "YOLOv13-S", "weights": "yolov13s.pt"},
    {"tag": "YOLO26n", "weights": "yolo26n.pt"},
    {"tag": "YOLO26s", "weights": "yolo26s.pt"},
]

print("WORKDIR:", WORKDIR)
print("Python:", sys.version)
print("Platform:", platform.platform())


WORKDIR: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2
Python: 3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 18:56:02) [GCC 11.2.0]
Platform: Linux-6.14.0-37-generic-x86_64-with-glibc2.39



## 1. Install dependencies

Nếu một vài model mới như YOLO12/YOLO26 không load được, update `ultralytics` lên bản mới nhất hoặc cài trực tiếp từ GitHub main.


In [8]:
import sys
import subprocess
import importlib.util

def is_installed(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def pip_install_minimal(packages):
    if not packages:
        print("All required packages already installed. No pip install needed.")
        return
    cmd = [sys.executable, "-m", "pip", "install"] + packages
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

# Minimal required packages only.
# Do NOT install/upgrade torch, numpy, opencv, pillow, pandas, sklearn, matplotlib here.
missing = []

if not is_installed("ultralytics"):
    missing.append("ultralytics")

if not is_installed("kagglehub"):
    missing.append("kagglehub")

pip_install_minimal(missing)

import torch
import ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

All required packages already installed. No pip install needed.
Ultralytics: 8.4.67
Torch: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
CUDA capability: (8, 9)
VRAM GB: 23.51



### Notes on compatibility

- Nếu `yolov13n.pt` hoặc `yolov13s.pt` không auto-download/load được, hãy cài YOLOv13 repo hoặc thay weights bằng YAML tương ứng để train-from-scratch. Với benchmark 30 epochs, pretrained `.pt` vẫn là ưu tiên.
- Nếu `yolo26*.pt` chưa load được, update Ultralytics lên bản mới nhất hoặc cài từ GitHub main.
- Notebook vẫn `try/except` từng model: model nào lỗi sẽ ghi vào bảng `errors.csv` và tiếp tục model tiếp theo, không làm hỏng toàn bộ run.



## 2. Download Kaggle dataset

Expected structure has nested `Root/Annotated Diseased/.../1. BG/images labels`, `2. WSSV/images labels`, `4. WSSV_BG/images labels`, and `Raw Images/Healthy`.


In [9]:

import kagglehub
from pathlib import Path

dataset_cache_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("KaggleHub dataset path:", dataset_cache_path)

LOCAL_RAW = WORKDIR / "kaggle_raw" / "shrimpdiseaseimagebd"
if not LOCAL_RAW.exists():
    LOCAL_RAW.parent.mkdir(parents=True, exist_ok=True)
    print("Copying dataset into:", LOCAL_RAW)
    shutil.copytree(dataset_cache_path, LOCAL_RAW)
else:
    print("Local raw copy already exists:", LOCAL_RAW)

def print_tree(root: Path, max_depth=4, max_items=220):
    count = 0
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        depth = len(rel.parts)
        if depth <= max_depth:
            print("  " * (depth-1) + ("[D] " if p.is_dir() else "[F] ") + p.name)
            count += 1
            if count >= max_items:
                print("... truncated ...")
                break

print_tree(LOCAL_RAW)


KaggleHub dataset path: /home/drnguyenvinh/.cache/kagglehub/datasets/nhanayai/shrimpdiseaseimagebd/versions/1
Local raw copy already exists: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2/kaggle_raw/shrimpdiseaseimagebd
[D] ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh
  [D] Root
    [D] Annotated Diseased Shrimp Images
      [D] Annotated Diseased Shrimp Images
    [D] Raw Images
      [D] Raw Images
    [F] Readme.docx
    [F] ShrimpDiseaseBD_Summary.xlsx



## 3. Locate folders automatically


In [ ]:
from pathlib import Path

IMG_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP"
}

def norm_name(s: str) -> str:
    return (
        str(s)
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
    )

def list_images(p: Path):
    p = Path(p)
    if not p.exists():
        return []
    return sorted([x for x in p.iterdir() if x.is_file() and x.suffix in IMG_EXTS])

def list_label_txts(p: Path):
    p = Path(p)
    if not p.exists():
        return []
    return sorted([x for x in p.iterdir() if x.is_file() and x.suffix.lower() == ".txt"])

def find_child_dir(parent: Path, wanted_names):
    wanted = {norm_name(x) for x in wanted_names}
    for child in Path(parent).iterdir():
        if child.is_dir() and norm_name(child.name) in wanted:
            return child
    return None

def group_key_from_name(name: str):
    n = norm_name(name)

    # Examples:
    # "1. BG" -> "1bg"
    # "2. WSSV" -> "2wssv"
    # "4. WSSV_BG" -> "4wssvbg"
    if n in {"1bg", "bg", "blackgill", "blackgillbg"}:
        return "BG"

    if n in {"2wssv", "wssv", "whitespot", "whitespotsyndromevirus"}:
        return "WSSV"

    if n in {
        "4wssvbg", "wssvbg", "bgwssv",
        "combinationofbgandwssv", "combinationbgwssv"
    }:
        return "WSSV_BG"

    return None

def find_disease_group_dirs(root: Path):
    """
    Finds annotated OD folders only:
      1. BG/images + labels
      2. WSSV/images + labels
      4. WSSV_BG/images + labels

    This function intentionally ignores Healthy because the current protocol is:
      - OD model trains only disease region detection
      - Healthy will be handled later by classification gate/head
    """
    root = Path(root)
    candidates = []

    for d in root.rglob("*"):
        if not d.is_dir():
            continue

        key = group_key_from_name(d.name)
        if key is None:
            continue

        img_dir = find_child_dir(d, ["images", "Images"])
        lab_dir = find_child_dir(d, ["labels", "Labels"])

        if img_dir is None or lab_dir is None:
            continue

        n_img = len(list_images(img_dir))
        n_lab = len(list_label_txts(lab_dir))

        if n_img > 0 and n_lab > 0:
            candidates.append((key, d, img_dir, lab_dir, n_img, n_lab))

    return candidates

# Find annotated disease folders
disease_dirs = find_disease_group_dirs(LOCAL_RAW)

print("Found annotation group dirs:")
for key, d, img, lab, n_img, n_lab in disease_dirs:
    print(f" - {key:7s} | {d} | images: {n_img} | labels: {n_lab}")

# Select the largest valid candidate for each disease group
group_dirs = {}
for key, d, img, lab, n_img, n_lab in disease_dirs:
    old = group_dirs.get(key)
    if old is None or n_img > old["n_images"]:
        group_dirs[key] = {
            "root": d,
            "images": img,
            "labels": lab,
            "n_images": n_img,
            "n_labels": n_lab,
        }

required = {"BG", "WSSV", "WSSV_BG"}
missing = required - set(group_dirs)
if missing:
    raise RuntimeError(
        f"Missing annotation groups: {missing}\n"
        f"Found groups: {sorted(group_dirs.keys())}\n"
        f"LOCAL_RAW = {LOCAL_RAW}"
    )

# Healthy is intentionally disabled for pure OD benchmark
USE_HEALTHY_NEGATIVES = False
HEALTHY_DIR = None

print("\nSelected disease dirs for pure OD training:")
for k in ["BG", "WSSV", "WSSV_BG"]:
    v = group_dirs[k]
    print(
        f"{k:7s} => root={v['root']} | "
        f"images={v['n_images']} | labels={v['n_labels']}"
    )

print("\nHealthy handling:")
print(" - USE_HEALTHY_NEGATIVES =", USE_HEALTHY_NEGATIVES)
print(" - Healthy folder search is skipped.")
print(" - Healthy will be handled later by a classification gate/head, not by OD training.")

Found annotation group dirs:
 - WSSV | /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2/kaggle_raw/shrimpdiseaseimagebd/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Annotated Diseased Shrimp Images/Annotated Diseased Shrimp Images/2. WSSV | images: 328 | labels: 328
 - BG | /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2/kaggle_raw/shrimpdiseaseimagebd/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Annotated Diseased Shrimp Images/Annotated Diseased Shrimp Images/1. BG | images: 198 | labels: 198
 - WSSV_BG | /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2/kaggle_raw/shrimpdiseaseimagebd/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Annotated Diseased Shrimp Images/Annotated Diseased

RuntimeError: Could not find Healthy folder with images.


## 4. Convert labels to canonical YOLO dataset

Remap:

- `BG` folder class `0` → canonical `BG=0`
- `WSSV` folder class `0` → canonical `WSSV=1`
- `WSSV_BG` folder class `0` → canonical `WSSV=1`
- `WSSV_BG` folder class `1` → canonical `BG=0`


In [ ]:

import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from PIL import Image

def remap_label(source_group: str, source_cls: int) -> int:
    if source_group == "BG":
        if source_cls != 0:
            raise ValueError(f"Unexpected BG class: {source_cls}")
        return 0
    if source_group == "WSSV":
        if source_cls != 0:
            raise ValueError(f"Unexpected WSSV class: {source_cls}")
        return 1
    if source_group == "WSSV_BG":
        if source_cls == 0:
            return 1
        if source_cls == 1:
            return 0
        raise ValueError(f"Unexpected WSSV_BG class: {source_cls}")
    raise ValueError(source_group)

def read_label_lines(label_path: Path, source_group: str):
    out = []
    raw = label_path.read_text().strip()
    if not raw:
        return out
    for line in raw.splitlines():
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Malformed line in {label_path}: {line}")
        cls = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:])
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid bbox in {label_path}: {line}")
        new_cls = remap_label(source_group, cls)
        out.append((new_cls, x, y, w, h))
    return out

records = []

for group in ["BG", "WSSV", "WSSV_BG"]:
    img_dir = group_dirs[group]["images"]
    lab_dir = group_dirs[group]["labels"]
    for img_path in list_images(img_dir):
        lab_path = lab_dir / f"{img_path.stem}.txt"
        if not lab_path.exists():
            raise FileNotFoundError(f"Missing label for {img_path}")
        boxes = read_label_lines(lab_path, group)
        records.append({
            "src_image": str(img_path),
            "src_label": str(lab_path),
            "source_group": group,
            "image_label": group,
            "is_negative": False,
            "n_boxes": len(boxes),
        })

for img_path in list_images(HEALTHY_DIR):
    records.append({
        "src_image": str(img_path),
        "src_label": "",
        "source_group": "Healthy",
        "image_label": "Healthy",
        "is_negative": True,
        "n_boxes": 0,
    })

df = pd.DataFrame(records)
print(df.groupby("image_label").size())
print("Total images:", len(df))

random.seed(SEED)
split_rows = []
for label, g in df.groupby("image_label", sort=True):
    idxs = list(g.index)
    random.shuffle(idxs)
    n = len(idxs)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    for i, idx in enumerate(idxs):
        split = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
        split_rows.append((idx, split))
split_map = dict(split_rows)
df["split"] = df.index.map(split_map)

if DATASET_OUT.exists():
    print("Dataset output exists; reusing:", DATASET_OUT)
else:
    for split in ["train", "val", "test"]:
        (DATASET_OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATASET_OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

    for _, r in df.iterrows():
        split = r["split"]
        src_img = Path(r["src_image"])
        dst_img = DATASET_OUT / "images" / split / src_img.name
        dst_lab = DATASET_OUT / "labels" / split / f"{src_img.stem}.txt"

        shutil.copy2(src_img, dst_img)

        if r["is_negative"]:
            dst_lab.write_text("")
        else:
            boxes = read_label_lines(Path(r["src_label"]), r["source_group"])
            text = "\n".join([f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in boxes])
            dst_lab.write_text(text + ("\n" if text else ""))

df.to_csv(DATASET_OUT / "image_metadata.csv", index=False)

data_yaml = {
    "path": str(DATASET_OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 2,
    "names": {0: "BG", 1: "WSSV"},
}
with open(DATASET_OUT / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(pd.crosstab(df["split"], df["image_label"]))
print("Prepared data.yaml:", DATASET_OUT / "data.yaml")



## 5. Dataset sanity checks


In [ ]:

def validate_yolo_dataset(root: Path):
    issues = []
    rows = []
    for split in ["train", "val", "test"]:
        img_dir = root / "images" / split
        lab_dir = root / "labels" / split
        for img in list_images(img_dir):
            lab = lab_dir / f"{img.stem}.txt"
            if not lab.exists():
                issues.append({"split": split, "image": str(img), "issue": "missing_label_file"})
                continue
            text = lab.read_text().strip()
            n_boxes = 0
            if text:
                for ln, line in enumerate(text.splitlines(), 1):
                    parts = line.split()
                    if len(parts) != 5:
                        issues.append({"split": split, "image": str(img), "issue": f"malformed_line_{ln}"})
                        continue
                    c = int(float(parts[0]))
                    x, y, w, h = map(float, parts[1:])
                    if c not in [0, 1]:
                        issues.append({"split": split, "image": str(img), "issue": f"bad_class_{c}"})
                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                        issues.append({"split": split, "image": str(img), "issue": f"bad_bbox_{line}"})
                    n_boxes += 1
            rows.append({"split": split, "image": img.name, "n_boxes": n_boxes})
    return pd.DataFrame(rows), pd.DataFrame(issues)

label_stats, label_issues = validate_yolo_dataset(DATASET_OUT)
label_stats.to_csv(REPORTS_DIR / "prepared_label_stats.csv", index=False)
label_issues.to_csv(REPORTS_DIR / "prepared_label_issues.csv", index=False)

print(label_stats.groupby("split").agg(images=("image", "count"), boxes=("n_boxes", "sum"), empty=("n_boxes", lambda s: int((s==0).sum()))))
print("Issues:", len(label_issues))
display(label_issues.head())



## 6. Training/evaluation utilities


In [ ]:

from ultralytics import YOLO
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd

DATA_YAML = str(DATASET_OUT / "data.yaml")
META_CSV = DATASET_OUT / "image_metadata.csv"

def safe_model_tag(tag: str):
    return tag.replace("/", "_").replace(" ", "_").replace("-", "").replace(".", "").replace(">", "").replace(":", "").lower()

def get_run_dir(tag: str):
    return RUNS_DIR / f"{safe_model_tag(tag)}_img{IMG_SIZE}_ep{EPOCHS}_seed{SEED}"

def extract_metrics_from_val(val_result):
    out = {}
    box = getattr(val_result, "box", None)
    if box is not None:
        for attr, key in [("mp", "precision_mean"), ("mr", "recall_mean"), ("map50", "map50"), ("map", "map50_95")]:
            try:
                out[key] = float(getattr(box, attr))
            except Exception:
                out[key] = np.nan
        try:
            maps = list(map(float, box.maps))
            out["map50_95_BG"] = maps[0] if len(maps) > 0 else np.nan
            out["map50_95_WSSV"] = maps[1] if len(maps) > 1 else np.nan
        except Exception:
            out["map50_95_BG"] = np.nan
            out["map50_95_WSSV"] = np.nan

    speed = getattr(val_result, "speed", None)
    if isinstance(speed, dict):
        for k, v in speed.items():
            try:
                out[f"val_speed_{k}_ms"] = float(v)
            except Exception:
                pass

    rd = getattr(val_result, "results_dict", None)
    if isinstance(rd, dict):
        for k, v in rd.items():
            kk = str(k).replace("metrics/", "").replace("(", "").replace(")", "").replace("/", "_").replace(" ", "_")
            try:
                out[f"raw_{kk}"] = float(v)
            except Exception:
                pass
    return out

def model_size_mb(path: Path):
    return path.stat().st_size / (1024**2) if Path(path).exists() else np.nan

def count_params_from_yolo(model_obj):
    try:
        return sum(p.numel() for p in model_obj.model.parameters()) / 1e6
    except Exception:
        return np.nan

DIAG_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]

def canonical_gt_from_image_label(label):
    return {"Healthy": 0, "BG": 1, "WSSV": 2, "WSSV_BG": 3}[label]

def diagnosis_from_pred_classes(classes_present):
    classes_present = set(int(c) for c in classes_present)
    has_bg = 0 in classes_present
    has_wssv = 1 in classes_present
    if has_bg and has_wssv:
        return 3
    if has_bg:
        return 1
    if has_wssv:
        return 2
    return 0

def evaluate_image_level_diagnosis(best_pt: Path, split="test", conf=PRED_CONF, iou=PRED_IOU):
    meta = pd.read_csv(META_CSV)
    meta_split = meta[meta["split"] == split].copy()
    img_dir = DATASET_OUT / "images" / split

    model = YOLO(str(best_pt))
    y_true, y_pred = [], []
    speeds, n_boxes_pred = [], []

    for img_name, label in zip(meta_split["src_image"].map(lambda x: Path(x).name), meta_split["image_label"]):
        img_path = img_dir / img_name
        results = model.predict(source=str(img_path), imgsz=IMG_SIZE, conf=conf, iou=iou, device=DEVICE, verbose=False, max_det=MAX_DET)
        r = results[0]
        pred_cls = []
        if r.boxes is not None and len(r.boxes) > 0:
            pred_cls = r.boxes.cls.detach().cpu().numpy().astype(int).tolist()
        y_true.append(canonical_gt_from_image_label(label))
        y_pred.append(diagnosis_from_pred_classes(pred_cls))
        n_boxes_pred.append(len(pred_cls))
        if isinstance(getattr(r, "speed", None), dict):
            speeds.append(r.speed)

    report = classification_report(y_true, y_pred, labels=[0,1,2,3], target_names=DIAG_NAMES, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    y_true_arr, y_pred_arr = np.array(y_true), np.array(y_pred)
    healthy_mask = y_true_arr == 0
    diseased_mask = y_true_arr != 0

    out = {
        "diag_accuracy": float(accuracy_score(y_true, y_pred)),
        "diag_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "diag_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "healthy_fp_rate": float(np.mean(y_pred_arr[healthy_mask] != 0)) if healthy_mask.any() else np.nan,
        "disease_miss_rate": float(np.mean(y_pred_arr[diseased_mask] == 0)) if diseased_mask.any() else np.nan,
        "diag_recall_Healthy": float(report["Healthy"]["recall"]),
        "diag_recall_BG": float(report["BG"]["recall"]),
        "diag_recall_WSSV": float(report["WSSV"]["recall"]),
        "diag_recall_WSSV_BG": float(report["WSSV_BG"]["recall"]),
        "avg_pred_boxes_per_image": float(np.mean(n_boxes_pred)),
    }

    if speeds:
        for k in speeds[0].keys():
            vals = [s.get(k, np.nan) for s in speeds]
            out[f"predict_speed_{k}_ms"] = float(np.nanmean(vals))
        inf = out.get("predict_speed_inference_ms", np.nan)
        out["predict_fps_from_inference"] = 1000.0 / inf if inf and inf > 0 else np.nan

    return out, cm, pd.DataFrame({"y_true": y_true, "y_pred": y_pred})

def benchmark_inference_speed(best_pt: Path, split="test", n_images=128, warmup=10, conf=PRED_CONF):
    meta = pd.read_csv(META_CSV)
    img_dir = DATASET_OUT / "images" / split
    image_names = meta[meta["split"] == split]["src_image"].map(lambda x: Path(x).name).tolist()
    image_paths = [str(img_dir / n) for n in image_names[:n_images]]
    if not image_paths:
        return {}

    model = YOLO(str(best_pt))
    for p in image_paths[:min(warmup, len(image_paths))]:
        _ = model.predict(source=p, imgsz=IMG_SIZE, conf=conf, device=DEVICE, verbose=False, max_det=MAX_DET)

    t0 = time.perf_counter()
    speed_rows = []
    for p in image_paths:
        res = model.predict(source=p, imgsz=IMG_SIZE, conf=conf, device=DEVICE, verbose=False, max_det=MAX_DET)
        r = res[0]
        if isinstance(getattr(r, "speed", None), dict):
            speed_rows.append(r.speed)
    wall = time.perf_counter() - t0

    out = {"bench_images": len(image_paths), "bench_wall_sec": wall, "bench_wall_fps": len(image_paths) / wall if wall > 0 else np.nan}
    if speed_rows:
        keys = sorted(set().union(*[x.keys() for x in speed_rows]))
        for k in keys:
            vals = [x.get(k, np.nan) for x in speed_rows]
            out[f"bench_{k}_ms_mean"] = float(np.nanmean(vals))
        inf = out.get("bench_inference_ms_mean", np.nan)
        out["bench_fps_from_inference_ms"] = 1000.0 / inf if inf and inf > 0 else np.nan
    return out



## 7. Train all models

Set `RUN_TRAINING = True`. The loop saves `model_comparison_live.csv` after every model, so interruption does not lose finished results.


In [ ]:

RUN_TRAINING = True

summary_rows = []
errors = []

if RUN_TRAINING:
    for spec in MODEL_SPECS:
        tag = spec["tag"]
        weights = spec["weights"]
        run_dir = get_run_dir(tag)
        run_dir.mkdir(parents=True, exist_ok=True)

        best_pt = run_dir / "weights" / "best.pt"
        last_pt = run_dir / "weights" / "last.pt"

        row = {
            "tag": tag,
            "weights": weights,
            "run_dir": str(run_dir),
            "status": "started",
            "start_time": datetime.now().isoformat(timespec="seconds"),
        }

        print("\n" + "="*100)
        print(f"MODEL: {tag} ({weights})")
        print("Run dir:", run_dir)
        print("="*100)

        try:
            if best_pt.exists():
                print("Found best.pt; skipping training and evaluating only.")
                eval_model_path = best_pt
            elif last_pt.exists():
                print("Found last.pt; resuming interrupted training.")
                model = YOLO(str(last_pt))
                _ = model.train(resume=True)
                eval_model_path = best_pt if best_pt.exists() else last_pt
            else:
                model = YOLO(weights)
                _ = model.train(
                    data=DATA_YAML,
                    epochs=EPOCHS,
                    imgsz=IMG_SIZE,
                    patience=PATIENCE,
                    batch=BATCH,
                    device=DEVICE,
                    workers=WORKERS,
                    seed=SEED,
                    deterministic=DETERMINISTIC,
                    pretrained=True,
                    optimizer="auto",
                    cos_lr=True,
                    close_mosaic=CLOSE_MOSAIC,
                    mosaic=1.0,
                    mixup=0.0,
                    copy_paste=0.0,
                    degrees=5.0,
                    translate=0.05,
                    scale=0.30,
                    shear=0.0,
                    perspective=0.0,
                    fliplr=0.5,
                    flipud=0.0,
                    hsv_h=0.015,
                    hsv_s=0.50,
                    hsv_v=0.30,
                    cache=CACHE,
                    amp=AMP,
                    project=str(RUNS_DIR),
                    name=run_dir.name,
                    exist_ok=True,
                    plots=True,
                    save=True,
                    save_period=-1,
                    val=True,
                    max_det=MAX_DET,
                )
                eval_model_path = best_pt if best_pt.exists() else last_pt

            if not eval_model_path.exists():
                raise FileNotFoundError(f"No checkpoint found for evaluation: {eval_model_path}")

            val_result = YOLO(str(eval_model_path)).val(
                data=DATA_YAML,
                split="test",
                imgsz=IMG_SIZE,
                batch=16,
                device=DEVICE,
                conf=0.001,
                iou=0.65,
                max_det=MAX_DET,
                plots=True,
                project=str(RUNS_DIR / "eval_test"),
                name=f"{run_dir.name}_test",
                exist_ok=True,
            )
            row.update(extract_metrics_from_val(val_result))

            diag_metrics, cm, pred_df = evaluate_image_level_diagnosis(eval_model_path, split="test", conf=PRED_CONF, iou=PRED_IOU)
            row.update(diag_metrics)
            pd.DataFrame(cm, index=DIAG_NAMES, columns=DIAG_NAMES).to_csv(REPORTS_DIR / f"{safe_model_tag(tag)}_confusion_matrix.csv")
            pred_df.to_csv(REPORTS_DIR / f"{safe_model_tag(tag)}_image_level_predictions.csv", index=False)

            row.update(benchmark_inference_speed(eval_model_path, split="test", n_images=128, warmup=10, conf=PRED_CONF))

            row["checkpoint_path"] = str(eval_model_path)
            row["model_size_mb"] = model_size_mb(eval_model_path)
            try:
                loaded = YOLO(str(eval_model_path))
                row["params_m"] = count_params_from_yolo(loaded)
            except Exception:
                row["params_m"] = np.nan

            if DO_EXPORT:
                export_dir = EXPORTS_DIR / safe_model_tag(tag)
                export_dir.mkdir(parents=True, exist_ok=True)
                for fmt in EXPORT_FORMATS:
                    try:
                        exported = YOLO(str(eval_model_path)).export(format=fmt, imgsz=IMG_SIZE, half=False, simplify=True)
                        row[f"export_{fmt}"] = str(exported)
                    except Exception as e:
                        row[f"export_{fmt}_error"] = repr(e)

            row["status"] = "ok"
            row["end_time"] = datetime.now().isoformat(timespec="seconds")

        except Exception as e:
            row["status"] = "failed"
            row["error"] = repr(e)
            row["end_time"] = datetime.now().isoformat(timespec="seconds")
            errors.append(row)
            print("FAILED:", tag, repr(e))

        summary_rows.append(row)
        pd.DataFrame(summary_rows).to_csv(REPORTS_DIR / "model_comparison_live.csv", index=False)
        with open(REPORTS_DIR / "model_comparison_live.json", "w") as f:
            json.dump(summary_rows, f, indent=2)

summary = pd.DataFrame(summary_rows)
summary.to_csv(REPORTS_DIR / "model_comparison_live.csv", index=False)
display(summary)



## 8. Final comparison table and mobile/edge score


In [ ]:

summary_csv = REPORTS_DIR / "model_comparison_live.csv"
if not summary_csv.exists():
    raise FileNotFoundError("Run the training cell first.")

df_sum = pd.read_csv(summary_csv)

num_cols = [
    "map50", "map50_95", "precision_mean", "recall_mean",
    "map50_95_BG", "map50_95_WSSV",
    "diag_accuracy", "diag_macro_f1", "diag_kappa",
    "healthy_fp_rate", "disease_miss_rate",
    "diag_recall_BG", "diag_recall_WSSV", "diag_recall_WSSV_BG",
    "model_size_mb", "params_m",
    "bench_wall_fps", "bench_fps_from_inference_ms",
    "bench_inference_ms_mean",
]
for c in num_cols:
    if c in df_sum.columns:
        df_sum[c] = pd.to_numeric(df_sum[c], errors="coerce")

def minmax_score(s, higher_is_better=True):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series([np.nan]*len(s), index=s.index)
    mn, mx = s.min(), s.max()
    out = pd.Series([1.0]*len(s), index=s.index) if mx == mn else (s - mn) / (mx - mn)
    return out if higher_is_better else 1 - out

df_sum["score_map"] = minmax_score(df_sum.get("map50_95", pd.Series(np.nan, index=df_sum.index)), True)
df_sum["score_diag"] = minmax_score(df_sum.get("diag_macro_f1", pd.Series(np.nan, index=df_sum.index)), True)
df_sum["score_hfp"] = minmax_score(df_sum.get("healthy_fp_rate", pd.Series(np.nan, index=df_sum.index)), False)
df_sum["score_dmiss"] = minmax_score(df_sum.get("disease_miss_rate", pd.Series(np.nan, index=df_sum.index)), False)
df_sum["score_fps"] = minmax_score(df_sum.get("bench_wall_fps", pd.Series(np.nan, index=df_sum.index)), True)
df_sum["score_size"] = minmax_score(df_sum.get("model_size_mb", pd.Series(np.nan, index=df_sum.index)), False)

df_sum["mobile_edge_score"] = (
    0.30 * df_sum["score_map"].fillna(0) +
    0.25 * df_sum["score_diag"].fillna(0) +
    0.15 * df_sum["score_hfp"].fillna(0) +
    0.10 * df_sum["score_dmiss"].fillna(0) +
    0.10 * df_sum["score_fps"].fillna(0) +
    0.10 * df_sum["score_size"].fillna(0)
)

cols = [
    "tag", "weights", "status",
    "map50", "map50_95", "precision_mean", "recall_mean",
    "map50_95_BG", "map50_95_WSSV",
    "diag_accuracy", "diag_macro_f1", "diag_kappa",
    "diag_recall_BG", "diag_recall_WSSV", "diag_recall_WSSV_BG",
    "healthy_fp_rate", "disease_miss_rate",
    "params_m", "model_size_mb",
    "bench_wall_fps", "bench_inference_ms_mean", "bench_fps_from_inference_ms",
    "mobile_edge_score",
    "checkpoint_path", "error",
]
cols = [c for c in cols if c in df_sum.columns]
final = df_sum.sort_values(["status", "mobile_edge_score"], ascending=[True, False])[cols]

final_csv = REPORTS_DIR / "final_model_comparison.csv"
final_md = REPORTS_DIR / "final_model_comparison.md"
final.to_csv(final_csv, index=False)
final.to_markdown(final_md, index=False)

display(final)
print("Saved:", final_csv)
print("Saved:", final_md)



## 9. Pareto plots


In [ ]:

import matplotlib.pyplot as plt

plot_df = final.copy()
if "status" in plot_df.columns:
    plot_df = plot_df[plot_df["status"].eq("ok")]

def scatter_with_labels(x, y, xlabel, ylabel, title, out_path):
    if x not in plot_df.columns or y not in plot_df.columns:
        print("Missing columns:", x, y)
        return
    d = plot_df[[x, y, "tag"]].dropna()
    if d.empty:
        print("No data for:", x, y)
        return
    plt.figure(figsize=(8, 6))
    plt.scatter(d[x], d[y])
    for _, r in d.iterrows():
        plt.annotate(str(r["tag"]), (r[x], r[y]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()
    print("Saved:", out_path)

scatter_with_labels("bench_wall_fps", "map50_95", "Wall-clock FPS", "mAP50-95", "Detection Accuracy vs Inference Speed", REPORTS_DIR / "pareto_map_fps.png")
scatter_with_labels("model_size_mb", "diag_macro_f1", "Checkpoint size MB", "Image-level Macro-F1", "Diagnosis Macro-F1 vs Model Size", REPORTS_DIR / "pareto_diag_size.png")
scatter_with_labels("healthy_fp_rate", "disease_miss_rate", "Healthy FP Rate", "Disease Miss Rate", "Safety Trade-off", REPORTS_DIR / "healthy_fp_vs_disease_miss.png")



## 10. Next step

Sau khi có bảng benchmark detector, chọn top 1–3 model theo:
- `mAP50-95`
- `diag_macro_f1`
- `healthy_fp_rate`
- `disease_miss_rate`
- `bench_wall_fps`
- `model_size_mb`

Sau đó mới train classifier gate Healthy → skip detector.
